# 01. Single-Experiment Registry Walkthrough

This notebook demonstrates the simplest **registry-driven end-to-end path** in this framework: select **one predefined experiment** from the shipped registry, run a dry-run, run real execution, inspect outputs, inspect artifact lineage, and produce a final summary.

Scope note:
- This notebook covers **one predefined registry experiment only**.
- Later notebooks cover workbook workflows and designed/factorial studies.


## Assumptions and prerequisites

Before running, set real local paths in the setup cell for:
- dataset index CSV (`INDEX_CSV`)
- data root directory (`DATA_ROOT`)

This notebook writes all run artifacts to a notebook-specific folder:
`outputs/notebook_demo/01_single_experiment/`


In [1]:
from __future__ import annotations

import json
import shlex
import shutil
import sqlite3
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# =========================
# EDIT THESE PATHS
# =========================
PROJECT_ROOT = Path(r"d:\Khaled\Projects\VScode\Thesis")
INDEX_CSV = PROJECT_ROOT / "Data" / "processed" / "dataset_index.csv"
DATA_ROOT = PROJECT_ROOT / "Data"
REGISTRY_PATH = PROJECT_ROOT / "configs" / "decision_support_registry.json"

NOTEBOOK_ROOT = PROJECT_ROOT / "outputs" / "notebook_demo" / "01_single_experiment"
CACHE_DIR = NOTEBOOK_ROOT / "cache"
OUTPUT_ROOT = NOTEBOOK_ROOT / "campaign_outputs"
NOTEBOOK_REGISTRY_DIR = NOTEBOOK_ROOT / "registry"

for path in (NOTEBOOK_ROOT, CACHE_DIR, OUTPUT_ROOT, NOTEBOOK_REGISTRY_DIR):
    path.mkdir(parents=True, exist_ok=True)

decision_support_cli = shutil.which("thesisml-run-decision-support")
if decision_support_cli:
    DECISION_SUPPORT_CMD = [decision_support_cli]
else:
    DECISION_SUPPORT_CMD = [sys.executable, "-m", "Thesis_ML.cli.decision_support"]

settings = pd.DataFrame(
    [
        ("PROJECT_ROOT", str(PROJECT_ROOT)),
        ("INDEX_CSV", str(INDEX_CSV)),
        ("DATA_ROOT", str(DATA_ROOT)),
        ("REGISTRY_PATH", str(REGISTRY_PATH)),
        ("CACHE_DIR", str(CACHE_DIR)),
        ("OUTPUT_ROOT", str(OUTPUT_ROOT)),
        ("NOTEBOOK_REGISTRY_DIR", str(NOTEBOOK_REGISTRY_DIR)),
    ],
    columns=["setting", "value"],
)
display(settings)
print("Decision-support command prefix:", " ".join(shlex.quote(part) for part in DECISION_SUPPORT_CMD))


,setting,value
0,PROJECT_ROOT,d:\Khaled\Projects\VScode\Thesis
1,INDEX_CSV,d:\Khaled\Projects\VScode\Thesis\Data\processe...
2,DATA_ROOT,d:\Khaled\Projects\VScode\Thesis\Data
3,REGISTRY_PATH,d:\Khaled\Projects\VScode\Thesis\configs\decis...
4,CACHE_DIR,d:\Khaled\Projects\VScode\Thesis\outputs\noteb...
5,OUTPUT_ROOT,d:\Khaled\Projects\VScode\Thesis\outputs\noteb...
6,NOTEBOOK_REGISTRY_DIR,d:\Khaled\Projects\VScode\Thesis\outputs\noteb...


Decision-support command prefix: 'd:\Khaled\Projects\VScode\Thesis\.venv\Scripts\thesisml-run-decision-support.EXE'


In [2]:
def run(cmd: list[str], *, cwd: Path | None = None, check: bool = False) -> subprocess.CompletedProcess[str]:
    resolved_cmd = [str(part) for part in cmd]
    print("$ " + " ".join(shlex.quote(part) for part in resolved_cmd))
    result = subprocess.run(
        resolved_cmd,
        cwd=str(cwd or PROJECT_ROOT),
        text=True,
        capture_output=True,
    )
    print(f"[returncode] {result.returncode}")
    if result.stdout.strip():
        print("\n[stdout]\n" + result.stdout)
    if result.stderr.strip():
        print("\n[stderr]\n" + result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            "Command failed with return code "
            f"{result.returncode}: {' '.join(resolved_cmd)}"
        )
    return result


def safe_read_json(path: Path):
    target = Path(path)
    if not target.exists():
        print(f"[missing] {target}")
        return None
    try:
        return json.loads(target.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"[invalid json] {target}: {exc}")
        return None


def display_json(title: str, payload) -> None:
    display(Markdown(f"#### {title}"))
    if payload is None:
        print("<missing>")
        return
    print(json.dumps(payload, indent=2))


def preview_csv(path: Path, *, n: int = 10) -> pd.DataFrame | None:
    target = Path(path)
    if not target.exists():
        print(f"[missing] {target}")
        return None
    try:
        df = pd.read_csv(target)
    except Exception as exc:
        print(f"[csv read failed] {target}: {exc}")
        return None
    display(Markdown(f"#### CSV preview: `{target}`"))
    display(df.head(n))
    return df


def print_tree(root: Path, *, max_depth: int = 5, max_entries: int = 400) -> None:
    root = Path(root)
    print(f"Output tree under: {root}")
    if not root.exists():
        print("(missing)")
        return

    printed = 0
    total_seen = 0
    for item in sorted(root.rglob("*")):
        rel = item.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        total_seen += 1
        if printed >= max_entries:
            continue
        indent = "  " * (depth - 1)
        suffix = "/" if item.is_dir() else ""
        print(f"{indent}- {rel.name}{suffix}")
        printed += 1

    if total_seen > printed:
        print(f"... truncated ({total_seen - printed} more entries at shown depth)")


def list_campaign_manifests(output_root: Path) -> list[Path]:
    campaigns_dir = Path(output_root) / "campaigns"
    if not campaigns_dir.exists():
        return []
    return sorted(campaigns_dir.glob("*/campaign_manifest.json"), key=lambda p: p.stat().st_mtime)


def dedupe_paths(paths: list[Path]) -> list[Path]:
    seen: set[str] = set()
    result: list[Path] = []
    for path in sorted(paths, key=lambda p: str(p)):
        key = str(path.resolve())
        if key in seen:
            continue
        seen.add(key)
        result.append(path)
    return result


## Framework health check

Run the canonical acceptance smoke command before notebook execution to verify the operator path and shipped assets.


In [3]:
smoke_script = PROJECT_ROOT / "scripts" / "acceptance_smoke.py"
if not smoke_script.exists():
    raise FileNotFoundError(f"Acceptance smoke script not found: {smoke_script}")

health_check_result = run([sys.executable, str(smoke_script)], cwd=PROJECT_ROOT)
HEALTH_CHECK_OK = health_check_result.returncode == 0
if not HEALTH_CHECK_OK:
    raise RuntimeError(
        "Framework health check failed. Resolve acceptance smoke failures before continuing."
    )
print("Framework health check passed.")


$ 'd:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe' 'd:\Khaled\Projects\VScode\Thesis\scripts\acceptance_smoke.py'
[returncode] 0

[stdout]
Template compile check: non-runnable template (no enabled rows).
No additional workbook sample assets detected in templates/.
$ thesisml-workbook --output C:\Users\Khaled\AppData\Local\Temp\thesisml-acceptance-b334a7mp\generated_workbook.xlsx
Created workbook: C:\Users\Khaled\AppData\Local\Temp\thesisml-acceptance-b334a7mp\generated_workbook.xlsx
Sheet order valid: True
Missing required sheets: None
Legacy required sheets present: True
New sheets present: True
Sheet count: 32
Data validations found: 96
Experiment_Definitions columns valid: True
Run_Log new columns present: True
Required named lists present: True
Missing named lists: None
Experiment_Ready formula present: True
Confirmatory formulas present: True
Dashboard formulas present: True
Stage vocabulary consistent: True
Stage rows detected: 7
$ thesisml-run-decision-support --regis

## Inspect shipped registry and choose one experiment

Load the shipped registry, inspect available experiments, and set one editable experiment ID.


In [4]:
registry_payload = safe_read_json(REGISTRY_PATH)
if not isinstance(registry_payload, dict):
    raise RuntimeError(f"Invalid registry payload at: {REGISTRY_PATH}")

experiments = list(registry_payload.get("experiments", []))
if not experiments:
    raise RuntimeError(f"No experiments found in registry: {REGISTRY_PATH}")

registry_rows = []
for exp in experiments:
    templates = list(exp.get("variant_templates", []))
    supported_count = sum(1 for item in templates if bool(item.get("supported", True)))
    registry_rows.append(
        {
            "experiment_id": exp.get("experiment_id"),
            "title": exp.get("title"),
            "stage": exp.get("stage"),
            "primary_metric": exp.get("primary_metric"),
            "execution_status": exp.get("execution_status"),
            "executable_now": exp.get("executable_now"),
            "variant_templates": len(templates),
            "supported_templates": supported_count,
        }
    )

registry_table = pd.DataFrame(registry_rows).sort_values("experiment_id").reset_index(drop=True)
display(registry_table)
print(f"Registry schema_version: {registry_payload.get('schema_version')}")
print(f"Registry path: {REGISTRY_PATH}")

# EDIT THIS VALUE
EXPERIMENT_ID = "E02"

selected_experiment = next(
    (exp for exp in experiments if str(exp.get("experiment_id")) == EXPERIMENT_ID),
    None,
)
if selected_experiment is None:
    valid = ", ".join(sorted(str(exp.get("experiment_id")) for exp in experiments))
    raise ValueError(f"Experiment '{EXPERIMENT_ID}' not found. Available IDs: {valid}")
print(f"Selected experiment ID: {EXPERIMENT_ID}")


,experiment_id,title,stage,primary_metric,execution_status,executable_now,variant_templates,supported_templates
0,E01,Target granularity experiment,Stage 1 - Target lock,balanced_accuracy,partially_blocked,True,3,2
1,E02,Task pooling experiment,Stage 1 - Target lock,balanced_accuracy,executable,True,2,2
2,E03,Modality pooling experiment,Stage 1 - Target lock,balanced_accuracy,executable,True,2,2
3,E04,Split-strength stress test,Stage 2 - Split lock,balanced_accuracy,partially_blocked,True,3,2
4,E05,Cross-person transfer design,Stage 2 - Split lock,balanced_accuracy,executable,True,1,1
5,E06,Linear model family comparison,Stage 3 - Model lock,balanced_accuracy,executable,True,1,1
6,E07,Class weighting experiment,Stage 3 - Model lock,balanced_accuracy,blocked,False,0,0
7,E08,Hyperparameter strategy experiment,Stage 3 - Model lock,balanced_accuracy,blocked,False,0,0
8,E09,Whole-brain versus ROI experiment,Stage 4 - Feature/preprocessing lock,balanced_accuracy,blocked,False,0,0
9,E10,Dimensionality reduction experiment,Stage 4 - Feature/preprocessing lock,balanced_accuracy,blocked,False,0,0


Registry schema_version: 1.0
Registry path: d:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json
Selected experiment ID: E02


## Create notebook-local single-experiment registry

To avoid accidental multi-experiment execution, write a notebook-local registry containing only the selected experiment.


In [5]:
single_registry_payload = dict(registry_payload)
single_registry_payload["experiments"] = [selected_experiment]

SINGLE_REGISTRY_PATH = NOTEBOOK_REGISTRY_DIR / f"single_experiment_{EXPERIMENT_ID}.json"
SINGLE_REGISTRY_PATH.write_text(
    json.dumps(single_registry_payload, indent=2) + "\n",
    encoding="utf-8",
)

print(f"Wrote single-experiment registry: {SINGLE_REGISTRY_PATH}")
display_json(
    "Single-experiment registry preview",
    {
        "schema_version": single_registry_payload.get("schema_version"),
        "description": single_registry_payload.get("description"),
        "experiment_ids": [
            exp.get("experiment_id") for exp in single_registry_payload.get("experiments", [])
        ],
    },
)


Wrote single-experiment registry: d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\registry\single_experiment_E02.json


#### Single-experiment registry preview

{
  "schema_version": "1.0",
  "description": "Decision-support experiment registry for thesis method locks (E01-E11).",
  "experiment_ids": [
    "E02"
  ]
}


## Explain the selected experiment before execution

Review the experiment intent and key configuration fields so the run scope is explicit before launching commands.


In [6]:
variant_templates = list(selected_experiment.get("variant_templates", []))
supported_templates = [item for item in variant_templates if bool(item.get("supported", True))]


def _collect_param_values(param_name: str) -> list[str]:
    values = set()
    for template in supported_templates:
        params = template.get("params", {})
        if not isinstance(params, dict):
            continue
        value = params.get(param_name)
        if value in (None, ""):
            continue
        values.add(str(value))
    return sorted(values)


def _as_text(values: list[str]) -> str:
    return ", ".join(values) if values else "<not explicitly fixed in params>"


experiment_summary = {
    "experiment_id": selected_experiment.get("experiment_id"),
    "title": selected_experiment.get("title"),
    "stage": selected_experiment.get("stage"),
    "manipulated_factor": selected_experiment.get("manipulated_factor"),
    "primary_metric": selected_experiment.get("primary_metric"),
    "secondary_metrics": selected_experiment.get("secondary_metrics"),
    "target": _as_text(_collect_param_values("target")),
    "model": _as_text(_collect_param_values("model")),
    "cv": _as_text(_collect_param_values("cv")),
    "filter_task": _as_text(_collect_param_values("filter_task")),
    "filter_modality": _as_text(_collect_param_values("filter_modality")),
    "dataset_scope": selected_experiment.get("dataset_scope"),
    "fixed_controls": selected_experiment.get("fixed_controls"),
    "notes": selected_experiment.get("notes"),
}

display(pd.DataFrame(experiment_summary.items(), columns=["field", "value"]))

template_rows = []
for template in variant_templates:
    params = template.get("params", {}) if isinstance(template.get("params"), dict) else {}
    template_rows.append(
        {
            "template_id": template.get("template_id"),
            "supported": template.get("supported", True),
            "target": params.get("target"),
            "model": params.get("model"),
            "cv": params.get("cv"),
            "filter_task": params.get("filter_task"),
            "filter_modality": params.get("filter_modality"),
            "unsupported_reason": template.get("unsupported_reason"),
        }
    )
display(pd.DataFrame(template_rows))


,field,value
0,experiment_id,E02
1,title,Task pooling experiment
2,stage,Stage 1 - Target lock
3,manipulated_factor,Pooling strategy by task
4,primary_metric,balanced_accuracy
5,secondary_metrics,"[macro_f1, accuracy]"
6,target,coarse_affect
7,model,ridge
8,cv,within_subject_loso_session
9,filter_task,<not explicitly fixed in params>


,template_id,supported,target,model,cv,filter_task,filter_modality,unsupported_reason
0,pooled_tasks,True,coarse_affect,ridge,within_subject_loso_session,None,None,None
1,task_specific,True,coarse_affect,ridge,within_subject_loso_session,None,None,None


## Dry-run the selected experiment

Dry-run validates the execution path and planning outputs without running full model fitting.


In [7]:
def build_campaign_command(*, dry_run: bool) -> list[str]:
    command = list(DECISION_SUPPORT_CMD) + [
        "--registry",
        str(SINGLE_REGISTRY_PATH),
        "--index-csv",
        str(INDEX_CSV),
        "--data-root",
        str(DATA_ROOT),
        "--cache-dir",
        str(CACHE_DIR),
        "--output-root",
        str(OUTPUT_ROOT),
        "--all",
    ]
    if dry_run:
        command.append("--dry-run")
    return command


existing_before_dry_run = set(list_campaign_manifests(OUTPUT_ROOT))
dry_run_command = build_campaign_command(dry_run=True)
dry_run_result = run(dry_run_command, cwd=PROJECT_ROOT)
DRY_RUN_OK = dry_run_result.returncode == 0

after_dry_run = set(list_campaign_manifests(OUTPUT_ROOT))
new_dry_run_manifests = sorted(
    [path for path in after_dry_run if path not in existing_before_dry_run],
    key=lambda p: p.stat().st_mtime,
)
if new_dry_run_manifests:
    DRY_RUN_CAMPAIGN_MANIFEST = new_dry_run_manifests[-1]
elif after_dry_run:
    DRY_RUN_CAMPAIGN_MANIFEST = sorted(after_dry_run, key=lambda p: p.stat().st_mtime)[-1]
else:
    DRY_RUN_CAMPAIGN_MANIFEST = None

print(f"Dry-run campaign manifest: {DRY_RUN_CAMPAIGN_MANIFEST}")
if not DRY_RUN_OK:
    raise RuntimeError("Dry-run failed. Stop here and fix errors before full execution.")


$ 'd:\Khaled\Projects\VScode\Thesis\.venv\Scripts\thesisml-run-decision-support.EXE' --registry 'd:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\registry\single_experiment_E02.json' --index-csv 'd:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv' --data-root 'd:\Khaled\Projects\VScode\Thesis\Data' --cache-dir 'd:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\cache' --output-root 'd:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\campaign_outputs' --all --dry-run
[returncode] 0

[stdout]
Campaign complete
- campaign_id: 20260314_143439_985342
- campaign_root: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\campaign_outputs\campaigns\20260314_143439_985342
- selected_experiments: E02
- status_counts: {'dry_run': 8}
- run_log_export: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\campaign_outputs\campaigns\20260314_143439_985342\run_log_e

## Run the experiment for real

Run the same command without `--dry-run` to perform real execution.


In [ ]:
existing_before_full_run = set(list_campaign_manifests(OUTPUT_ROOT))
full_run_command = build_campaign_command(dry_run=False)
full_run_result = run(full_run_command, cwd=PROJECT_ROOT)
FULL_RUN_OK = full_run_result.returncode == 0

after_full_run = set(list_campaign_manifests(OUTPUT_ROOT))
new_full_run_manifests = sorted(
    [path for path in after_full_run if path not in existing_before_full_run],
    key=lambda p: p.stat().st_mtime,
)
if new_full_run_manifests:
    FULL_RUN_CAMPAIGN_MANIFEST = new_full_run_manifests[-1]
elif after_full_run:
    FULL_RUN_CAMPAIGN_MANIFEST = sorted(after_full_run, key=lambda p: p.stat().st_mtime)[-1]
else:
    FULL_RUN_CAMPAIGN_MANIFEST = None

print(f"Full-run campaign manifest: {FULL_RUN_CAMPAIGN_MANIFEST}")
if not FULL_RUN_OK:
    raise RuntimeError("Full execution failed. Review command output above.")


$ 'd:\Khaled\Projects\VScode\Thesis\.venv\Scripts\thesisml-run-decision-support.EXE' --registry 'd:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\registry\single_experiment_E02.json' --index-csv 'd:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv' --data-root 'd:\Khaled\Projects\VScode\Thesis\Data' --cache-dir 'd:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\cache' --output-root 'd:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\01_single_experiment\campaign_outputs' --all


## Inspect outputs on disk

Show a readable output tree and highlight important files.


In [ ]:
REFERENCE_CAMPAIGN_MANIFEST_PATH = FULL_RUN_CAMPAIGN_MANIFEST or DRY_RUN_CAMPAIGN_MANIFEST
CAMPAIGN_MANIFEST = (
    safe_read_json(REFERENCE_CAMPAIGN_MANIFEST_PATH)
    if REFERENCE_CAMPAIGN_MANIFEST_PATH is not None
    else None
)

if CAMPAIGN_MANIFEST:
    display_json("Selected campaign manifest", CAMPAIGN_MANIFEST)
else:
    print("No campaign manifest could be loaded.")

search_roots = [OUTPUT_ROOT]
if isinstance(CAMPAIGN_MANIFEST, dict):
    campaign_root_text = str(CAMPAIGN_MANIFEST.get("campaign_root") or "").strip()
    if campaign_root_text:
        search_roots.append(Path(campaign_root_text))
    experiment_roots = CAMPAIGN_MANIFEST.get("experiment_roots", {})
    if isinstance(experiment_roots, dict):
        for root_text in experiment_roots.values():
            root_text = str(root_text or "").strip()
            if root_text:
                search_roots.append(Path(root_text))

search_roots = dedupe_paths([path for path in search_roots if Path(path).exists()])
print("Search roots:")
for root in search_roots:
    print(f"- {root}")

KEY_FILENAMES = [
    "run_status.json",
    "config.json",
    "metrics.json",
    "predictions.csv",
    "fold_metrics.csv",
    "spatial_compatibility_report.json",
    "decision_support_summary.csv",
    "summary_outputs_export.csv",
    "run_log_export.csv",
    "artifact_registry.sqlite3",
]

KEY_FILE_MAP = {}
for name in KEY_FILENAMES:
    found_paths = []
    for root in search_roots:
        found_paths.extend(root.rglob(name))
    KEY_FILE_MAP[name] = dedupe_paths(found_paths)

key_rows = []
for name in KEY_FILENAMES:
    matches = KEY_FILE_MAP.get(name, [])
    key_rows.append(
        {
            "filename": name,
            "found_count": len(matches),
            "example_path": str(matches[0]) if matches else "",
        }
    )
display(pd.DataFrame(key_rows))

print_tree(OUTPUT_ROOT, max_depth=6, max_entries=500)


## Inspect key JSON outputs

Load and display run status, metrics, and spatial compatibility report where present.


In [ ]:
JSON_PRIORITY = [
    "run_status.json",
    "metrics.json",
    "spatial_compatibility_report.json",
]

for filename in JSON_PRIORITY:
    candidates = KEY_FILE_MAP.get(filename, [])
    if not candidates:
        print(f"[not found] {filename}")
        continue
    payload = safe_read_json(candidates[0])
    display_json(f"{filename} ({candidates[0]})", payload)


## Inspect key CSV outputs

Preview predictions, fold metrics, and summary CSVs when available.


In [ ]:
CSV_PRIORITY = [
    "predictions.csv",
    "fold_metrics.csv",
    "decision_support_summary.csv",
    "summary_outputs_export.csv",
    "run_log_export.csv",
]

for filename in CSV_PRIORITY:
    candidates = KEY_FILE_MAP.get(filename, [])
    if not candidates:
        print(f"[not found] {filename}")
        continue
    preview_csv(candidates[0], n=8)


## Inspect artifact lineage (SQLite)

Artifact lineage matters for provenance and auditability: it records what artifacts were produced, their run IDs, and upstream dependencies used to create them.


In [ ]:
artifact_registry_candidates = list(KEY_FILE_MAP.get("artifact_registry.sqlite3", []))
if not artifact_registry_candidates and isinstance(CAMPAIGN_MANIFEST, dict):
    registry_path_text = str(CAMPAIGN_MANIFEST.get("artifact_registry_path") or "").strip()
    if registry_path_text:
        artifact_registry_candidates = [Path(registry_path_text)]

ARTIFACT_REGISTRY_PATH = artifact_registry_candidates[0] if artifact_registry_candidates else None
if ARTIFACT_REGISTRY_PATH is None or not Path(ARTIFACT_REGISTRY_PATH).exists():
    print("Artifact registry SQLite file not found for this run.")
else:
    ARTIFACT_REGISTRY_PATH = Path(ARTIFACT_REGISTRY_PATH)
    print(f"artifact_registry_path={ARTIFACT_REGISTRY_PATH}")
    with sqlite3.connect(str(ARTIFACT_REGISTRY_PATH)) as connection:
        tables_df = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
            connection,
        )
        display(Markdown("#### SQLite tables"))
        display(tables_df)

        table_names = set(tables_df["name"].astype(str).tolist())
        if "artifacts" in table_names:
            artifacts_df = pd.read_sql_query(
                """
                SELECT artifact_id, artifact_type, run_id, upstream_artifact_ids,
                       status, created_at, path
                FROM artifacts
                ORDER BY created_at DESC
                LIMIT 30
                """,
                connection,
            )
            display(Markdown("#### artifacts table preview"))
            display(artifacts_df)

            lineage_rows = []
            for _, row in artifacts_df.iterrows():
                raw_upstream = row.get("upstream_artifact_ids")
                upstream_ids = []
                if isinstance(raw_upstream, str) and raw_upstream.strip():
                    try:
                        parsed = json.loads(raw_upstream)
                        if isinstance(parsed, list):
                            upstream_ids = [str(item) for item in parsed]
                    except Exception:
                        upstream_ids = []

                if not upstream_ids:
                    lineage_rows.append(
                        {
                            "artifact_id": row.get("artifact_id"),
                            "upstream_artifact_id": None,
                            "artifact_type": row.get("artifact_type"),
                            "status": row.get("status"),
                        }
                    )
                    continue

                for upstream_id in upstream_ids:
                    lineage_rows.append(
                        {
                            "artifact_id": row.get("artifact_id"),
                            "upstream_artifact_id": upstream_id,
                            "artifact_type": row.get("artifact_type"),
                            "status": row.get("status"),
                        }
                    )

            lineage_df = pd.DataFrame(lineage_rows)
            display(Markdown("#### Lineage edges (artifact -> upstream artifact)"))
            display(lineage_df.head(40))
        else:
            print("No 'artifacts' table found in artifact registry database.")


## Final run summary

Build a compact summary with run status, important file counts, and key artifact paths.


In [ ]:
def first_path(filename: str) -> str:
    candidates = KEY_FILE_MAP.get(filename, [])
    return str(candidates[0]) if candidates else ""


important_unique_paths = sorted(
    {str(path) for paths in KEY_FILE_MAP.values() for path in paths}
)

FINAL_SUMMARY = {
    "selected_experiment_id": EXPERIMENT_ID,
    "registry_path_used": str(SINGLE_REGISTRY_PATH),
    "output_root": str(OUTPUT_ROOT),
    "dry_run_succeeded": bool(DRY_RUN_OK),
    "full_execution_succeeded": bool(FULL_RUN_OK),
    "important_files_found_count": int(len(important_unique_paths)),
    "key_result_paths": {
        "campaign_manifest": str(REFERENCE_CAMPAIGN_MANIFEST_PATH) if REFERENCE_CAMPAIGN_MANIFEST_PATH else "",
        "run_status": first_path("run_status.json"),
        "config": first_path("config.json"),
        "metrics": first_path("metrics.json"),
        "predictions": first_path("predictions.csv"),
        "fold_metrics": first_path("fold_metrics.csv"),
        "spatial_compatibility_report": first_path("spatial_compatibility_report.json"),
        "decision_support_summary": first_path("decision_support_summary.csv"),
        "artifact_registry": first_path("artifact_registry.sqlite3"),
    },
}

display(pd.DataFrame(FINAL_SUMMARY.items(), columns=["field", "value"]))
display_json("Final summary (JSON)", FINAL_SUMMARY)


## Next step

This notebook covered the simplest predefined registry experiment path.

Next planned notebook: `02_single_experiment_workbook_walkthrough.ipynb`.
